# MVS-XAI: Multi-View Stacking Ensemble with Explainable AI
## IEEE-CIS Pipeline (590K E-Commerce Fraud Records) — **Optimized v2**
---
**Full 8-Stage Architecture** — Tier 1+2 Optimizations Applied.

| Stage | Component | Status |
|-------|-----------|--------|
| 1 | Preprocessing (Null, Freq Encoding, Interaction FE, Min-Max) | ✅ |
| 2 | 5-Fold Stratified CV + Native Weights | ✅ |
| 3 | Dual-View: Tabular (130+ feat) + Sequential (T=10, 11 feat) | ✅ |
| 5 | Meta-Learner: Logistic Regression (L2) on OOF [Nx3] | ✅ |
| 6 | 4-Level XAI: SHAP, LIME, DiCE, Anchors | ✅ |
| 7 | Dual-Output: Real-Time + Audit Report | ✅ |


In [ ]:
# ====== MOUNT GOOGLE DRIVE ======
from google.colab import drive


### 📥 Hướng dẫn tải Dataset IEEE-CIS


In [ ]:
# ====== [OPTIONAL] DOWNLOAD DATASET FROM KAGGLE ======
# Chỉ chạy cell này nếu chưa có file trên Drive.
# Bước 1: Upload file kaggle.json (API key) lên Colab.
#   - Vào https://www.kaggle.com/settings -> Create New Token -> Tải kaggle.json
#   - Upload lên Colab qua sidebar Files
#
# Bước 2: Chạy cell này:

import os
# os.environ['KAGGLE_CONFIG_DIR'] = '/content'
# !pip install -q kaggle
# !kaggle competitions download -c ieee-fraud-detection -p /content/drive/MyDrive/ieee-cis-fraud-detection/
# !cd /content/drive/MyDrive/ieee-cis-fraud-detection/ && unzip -o ieee-fraud-detection.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# ====== REPO ROOT DISCOVERY ======
import sys
from pathlib import Path
_repo_candidates = [
    '/content/drive/MyDrive/FDB1-DoAn',
    '/content/drive/MyDrive/Digital Financial Transaction Fraud Detection Using Explainable Multi-Model Machine Learning on PaySim and IEEE-CIS',
    '/content/FDB1-DoAn',
]
REPO_ROOT = None
for _cand in _repo_candidates:
    if Path(_cand).exists():
        REPO_ROOT = _cand
        break
if REPO_ROOT is not None:
    os.chdir(REPO_ROOT)
    if REPO_ROOT not in sys.path:
        sys.path.insert(0, REPO_ROOT)
    print(f'Repo root set to: {REPO_ROOT}')
else:
    print('Repo root not found from default candidates; local benchmark modules may be unavailable.')

# ====== DEPENDENCY INSTALLATION ======
!pip install -q lightgbm xgboost imbalanced-learn shap lime dice-ml alibi networkx catboost
!pip install -q tensorflow

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import os, glob, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler, LabelEncoder, OrdinalEncoder
from sklearn.model_selection import TimeSeriesSplit, train_test_split
from sklearn.metrics import (classification_report, average_precision_score,
                             roc_auc_score, precision_score, recall_score, f1_score, brier_score_loss)

import lightgbm as lgb
import xgboost as xgb
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow import keras
import torch
import torch.nn as nn
import torch.nn.functional as F

import shap
import lime
import lime.lime_tabular
import dice_ml

print('All 8-Stage dependencies loaded successfully (Tier 2 Optimized).')


---
## STAGE 1: Data Preprocessing Pipeline
**Components:** Null handling → Encoding → Min-Max Scaling → Feature Engineering

IEEE-CIS has 434 features with >40% missing values. We apply:
- Categorical NaN → `UNKNOWN`
- Numerical NaN → Median imputation
- Merge `train_transaction.csv` + `train_identity.csv` on `TransactionID`


In [ ]:
def stage1_preprocessing(df):
    print('[Stage 1] Base Preprocessing Pipeline...')

    cat_cols = df.select_dtypes(include='object').columns.tolist()
    num_cols = df.select_dtypes(include=np.number).columns.tolist()
    num_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]

    for c in cat_cols:
        df[c] = df[c].fillna('UNKNOWN')
    # Numeric imputation is fit per fold to avoid global leakage.

    raw_cols = ['P_emaildomain', 'R_emaildomain', 'DeviceInfo', 'DeviceType',
                'id_30', 'id_31', 'id_33', 'ProductCD',
                'addr1', 'addr2', 'card1', 'card2', 'card3', 'card5']
    for c in raw_cols:
        if c in df.columns:
            df[f'{c}_raw'] = df[c].astype(str).copy()

    fp_parts = []
    for c in ['DeviceType_raw', 'DeviceInfo_raw', 'id_30_raw', 'id_31_raw', 'id_33_raw']:
        if c in df.columns:
            fp_parts.append(df[c].fillna('UNKNOWN'))
        else:
            fp_parts.append(pd.Series(['UNKNOWN'] * len(df), index=df.index))
    df['device_fp_raw'] = fp_parts[0] + '|' + fp_parts[1] + '|' + fp_parts[2] + '|' + fp_parts[3] + '|' + fp_parts[4]

    if 'TransactionDT' in df.columns:
        df = df.sort_values('TransactionDT').reset_index(drop=True)

    if 'TransactionDT' in df.columns and 'D1' in df.columns and 'card1_raw' in df.columns and 'addr1_raw' in df.columns:
        df['Day'] = (df['TransactionDT'] / 86400).astype(int)
        df['D1n'] = df['Day'] - df['D1'].fillna(0)
        df['AccountID'] = df['card1_raw'].astype(str) + '_' + df['addr1_raw'].astype(str) + '_' + df['D1n'].astype(str)
    elif 'card1_raw' in df.columns and 'card2_raw' in df.columns:
        df['AccountID'] = df['card1_raw'].astype(str) + '_' + df['card2_raw'].astype(str)
    elif 'card1' in df.columns and 'card2' in df.columns:
        df['AccountID'] = df['card1'].astype(str) + '_' + df['card2'].astype(str)
    else:
        df['AccountID'] = df.index.astype(str)

    if 'TransactionDT' in df.columns:
        df['Hour'] = (df['TransactionDT'] / 3600).astype(int) % 24
        df['DayOfWeek'] = (df['TransactionDT'] / 86400).astype(int) % 7

    if 'TransactionAmt' in df.columns:
        df['LogAmt'] = np.log1p(df['TransactionAmt'])
        df['Amt_cents'] = np.round(df['TransactionAmt'] - np.floor(df['TransactionAmt']), 2).astype(np.float32)

    if 'TransactionAmt' in df.columns and 'Hour' in df.columns:
        df['Amt_x_Hour'] = (df['TransactionAmt'] * df['Hour']).astype(np.float32)
    if 'C1' in df.columns and 'C14' in df.columns:
        df['C1_div_C14'] = (df['C1'] / (df['C14'] + 1e-6)).astype(np.float32)

    # Fold-local aggregate, temporal group, and graph features are computed inside CV.

    print('  [Tier 3] Fold-local encoding / target encoding / PCA / scaling will run inside CV.')
    print(f'  Base shape before fold-local transforms: {df.shape}')
    print(f'  Fraud ratio: {df["isFraud"].mean():.4%}')
    for col in df.select_dtypes(include=['float64']).columns:
        df[col] = df[col].astype(np.float32)
    import gc; gc.collect()
    print(f'  Memory after base preprocessing: {df.memory_usage(deep=True).sum()/1e9:.2f} GB')
    return df



def add_fold_local_aggregate_features(df_tr, df_vl):
    df_tr = df_tr.reset_index(drop=True).copy()
    df_vl = df_vl.reset_index(drop=True).copy()

    if 'TransactionAmt' in df_tr.columns:
        global_amt_mean = float(df_tr['TransactionAmt'].mean())
    else:
        global_amt_mean = 0.0

    if 'TransactionDT' in df_tr.columns and 'AccountID' in df_tr.columns:
        combo = pd.concat([
            df_tr.assign(_fold_split='tr', _fold_row=np.arange(len(df_tr))),
            df_vl.assign(_fold_split='vl', _fold_row=np.arange(len(df_vl))),
        ], ignore_index=True)
        combo = combo.sort_values('TransactionDT').reset_index(drop=True)
        combo['DT_diff'] = combo.groupby('AccountID')['TransactionDT'].diff().fillna(86400).astype(np.float32)
        if 'TransactionAmt' in combo.columns:
            grp_amt = combo.groupby('AccountID')['TransactionAmt']
            combo['Amt_mean_3'] = grp_amt.transform(lambda x: x.rolling(3, min_periods=1).mean()).astype(np.float32)
            combo['Amt_std_3'] = grp_amt.transform(lambda x: x.rolling(3, min_periods=1).std()).fillna(0).astype(np.float32)
        temporal_cols = [c for c in ['DT_diff', 'Amt_mean_3', 'Amt_std_3'] if c in combo.columns]
        tr_temporal = combo.loc[combo['_fold_split'] == 'tr', ['_fold_row'] + temporal_cols].sort_values('_fold_row')
        vl_temporal = combo.loc[combo['_fold_split'] == 'vl', ['_fold_row'] + temporal_cols].sort_values('_fold_row')
        for col in temporal_cols:
            df_tr[col] = tr_temporal[col].to_numpy(dtype=np.float32, copy=False)
            df_vl[col] = vl_temporal[col].to_numpy(dtype=np.float32, copy=False)

    if 'AccountID' in df_tr.columns:
        uid_size = df_tr.groupby('AccountID').size()
        df_tr['UID_count'] = df_tr['AccountID'].map(uid_size).fillna(0).astype(np.float32)
        df_vl['UID_count'] = df_vl['AccountID'].map(uid_size).fillna(0).astype(np.float32)

    if 'TransactionAmt' in df_tr.columns and 'AccountID' in df_tr.columns:
        uid_amt_mean = df_tr.groupby('AccountID')['TransactionAmt'].mean()
        uid_amt_std = df_tr.groupby('AccountID')['TransactionAmt'].std().fillna(0)
        df_tr['UID_Amt_mean'] = df_tr['AccountID'].map(uid_amt_mean).fillna(global_amt_mean).astype(np.float32)
        df_vl['UID_Amt_mean'] = df_vl['AccountID'].map(uid_amt_mean).fillna(global_amt_mean).astype(np.float32)
        df_tr['UID_Amt_std'] = df_tr['AccountID'].map(uid_amt_std).fillna(0).astype(np.float32)
        df_vl['UID_Amt_std'] = df_vl['AccountID'].map(uid_amt_std).fillna(0).astype(np.float32)

    col_list = df_tr.columns.tolist()
    if 'AccountID' in df_tr.columns and 'C1' in col_list:
        uid_c1_mean = df_tr.groupby('AccountID')['C1'].mean()
        df_tr['UID_C1_mean'] = df_tr['AccountID'].map(uid_c1_mean).fillna(0).astype(np.float32)
        df_vl['UID_C1_mean'] = df_vl['AccountID'].map(uid_c1_mean).fillna(0).astype(np.float32)
    if 'AccountID' in df_tr.columns and 'C14' in col_list:
        uid_c14_mean = df_tr.groupby('AccountID')['C14'].mean()
        df_tr['UID_C14_mean'] = df_tr['AccountID'].map(uid_c14_mean).fillna(0).astype(np.float32)
        df_vl['UID_C14_mean'] = df_vl['AccountID'].map(uid_c14_mean).fillna(0).astype(np.float32)
    if 'AccountID' in df_tr.columns and 'D1' in col_list:
        uid_d1_std = df_tr.groupby('AccountID')['D1'].std().fillna(0)
        df_tr['UID_D1_std'] = df_tr['AccountID'].map(uid_d1_std).fillna(0).astype(np.float32)
        df_vl['UID_D1_std'] = df_vl['AccountID'].map(uid_d1_std).fillna(0).astype(np.float32)

    for card_col in ['card1', 'card2', 'card5']:
        if card_col in df_tr.columns and 'TransactionAmt' in df_tr.columns:
            card_grp = df_tr.groupby(card_col)['TransactionAmt']
            count_map = card_grp.count()
            mean_map = card_grp.mean()
            std_map = card_grp.std().fillna(0)
            df_tr[f'{card_col}_count'] = df_tr[card_col].map(count_map).fillna(0).astype(np.float32)
            df_vl[f'{card_col}_count'] = df_vl[card_col].map(count_map).fillna(0).astype(np.float32)
            df_tr[f'{card_col}_amt_mean'] = df_tr[card_col].map(mean_map).fillna(global_amt_mean).astype(np.float32)
            df_vl[f'{card_col}_amt_mean'] = df_vl[card_col].map(mean_map).fillna(global_amt_mean).astype(np.float32)
            df_tr[f'{card_col}_amt_std'] = df_tr[card_col].map(std_map).fillna(0).astype(np.float32)
            df_vl[f'{card_col}_amt_std'] = df_vl[card_col].map(std_map).fillna(0).astype(np.float32)

    if 'TransactionAmt' in df_tr.columns and 'card1' in df_tr.columns:
        card1_mean = df_tr.groupby('card1')['TransactionAmt'].mean()
        tr_card_mean = df_tr['card1'].map(card1_mean).fillna(global_amt_mean)
        vl_card_mean = df_vl['card1'].map(card1_mean).fillna(global_amt_mean)
        df_tr['Amt_card_dev'] = (df_tr['TransactionAmt'] - tr_card_mean).astype(np.float32)
        df_vl['Amt_card_dev'] = (df_vl['TransactionAmt'] - vl_card_mean).astype(np.float32)

    return df_tr, df_vl


def fit_apply_fold_graph_features(df_tr, df_vl):
    if not globals().get('ENABLE_GRAPH_VIEW', True):
        return df_tr, df_vl
    combo = pd.concat([
        df_tr.assign(_fold_split='tr', _fold_row=np.arange(len(df_tr))),
        df_vl.assign(_fold_split='vl', _fold_row=np.arange(len(df_vl))),
    ], ignore_index=True)
    combo = combo.sort_values('TransactionDT').reset_index(drop=True)
    _, gcols = extract_graph_view(combo)
    tr_graph = combo.loc[combo['_fold_split'] == 'tr', ['_fold_row'] + gcols].sort_values('_fold_row')
    vl_graph = combo.loc[combo['_fold_split'] == 'vl', ['_fold_row'] + gcols].sort_values('_fold_row')
    for col in gcols:
        df_tr[col] = tr_graph[col].to_numpy(dtype=np.float32, copy=False)
        df_vl[col] = vl_graph[col].to_numpy(dtype=np.float32, copy=False)
    return df_tr, df_vl


def fit_apply_fold_preprocessing(df_tr, df_vl):
    import re
    from sklearn.decomposition import PCA

    df_tr = df_tr.reset_index(drop=True).copy()
    df_vl = df_vl.reset_index(drop=True).copy()
    print('    [Fold Prep] Train-only aggregate/graph/ordinal/TE/PCA/scaler...')

    num_cols_base = df_tr.select_dtypes(include=np.number).columns.tolist()
    num_cols_base = [c for c in num_cols_base if c not in ['TransactionID', 'isFraud']]
    if num_cols_base:
        medians = df_tr[num_cols_base].median()
        df_tr[num_cols_base] = df_tr[num_cols_base].fillna(medians)
        df_vl[num_cols_base] = df_vl[num_cols_base].fillna(medians)

    df_tr, df_vl = add_fold_local_aggregate_features(df_tr, df_vl)
    df_tr, df_vl = fit_apply_fold_graph_features(df_tr, df_vl)

    freq_cols = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'DeviceType', 'DeviceInfo']
    for fc in freq_cols:
        if fc in df_tr.columns:
            freq = df_tr[fc].value_counts(normalize=True)
            df_tr[f'{fc}_freq'] = df_tr[fc].map(freq).fillna(0).astype(np.float32)
            df_vl[f'{fc}_freq'] = df_vl[fc].map(freq).fillna(0).astype(np.float32)

    te_cols = ['card1', 'card2', 'card3', 'addr1', 'addr2', 'P_emaildomain']
    global_mean = float(df_tr['isFraud'].mean())

    def smooth_target_encoding(col, weight=300):
        if col not in df_tr.columns:
            return
        agg = df_tr.groupby(col)['isFraud'].agg(['count', 'mean'])
        smooth = (agg['count'] * agg['mean'] + weight * global_mean) / (agg['count'] + weight)
        mapping = smooth.to_dict()
        df_tr[f'{col}_fraud_rate'] = df_tr[col].map(mapping).fillna(global_mean).astype(np.float32)
        df_vl[f'{col}_fraud_rate'] = df_vl[col].map(mapping).fillna(global_mean).astype(np.float32)

    for c in te_cols:
        smooth_target_encoding(c, weight=300)

    v_cols = [c for c in df_tr.columns if re.match(r'^V\d+$', c)]
    if len(v_cols) > 20:
        pca = PCA(n_components=15, random_state=42)
        tr_v = pca.fit_transform(df_tr[v_cols].fillna(0))
        vl_v = pca.transform(df_vl[v_cols].fillna(0))
        for j in range(15):
            df_tr[f'V_pca_{j}'] = tr_v[:, j].astype(np.float32)
            df_vl[f'V_pca_{j}'] = vl_v[:, j].astype(np.float32)
        df_tr.drop(columns=v_cols, inplace=True)
        df_vl.drop(columns=v_cols, inplace=True)

    cat_cols = [
        c for c in df_tr.select_dtypes(include='object').columns
        if not c.endswith('_raw') and c not in ['device_fp_raw', 'AccountID']
    ]
    if cat_cols:
        oe = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        df_tr[cat_cols] = oe.fit_transform(df_tr[cat_cols].astype(str))
        df_vl[cat_cols] = oe.transform(df_vl[cat_cols].astype(str))

    num_cols = df_tr.select_dtypes(include=np.number).columns.tolist()
    scale_cols = [c for c in num_cols if c not in ['TransactionID', 'isFraud']]
    scaler = MinMaxScaler()
    df_tr[scale_cols] = scaler.fit_transform(df_tr[scale_cols])
    df_vl[scale_cols] = scaler.transform(df_vl[scale_cols])

    for frame in [df_tr, df_vl]:
        for col in frame.select_dtypes(include=['float64']).columns:
            frame[col] = frame[col].astype(np.float32)

    return df_tr, df_vl


---
## STAGE 3: Multi-View Feature Engineering
- **View 1 (Tabular):** Numeric + encoded categorical features
- **View 2 (Sequential):** T=10 sliding window per AccountID proxy


In [ ]:
def select_tabular_cols(df):
    """Select top ~130+ features: base + C/D/M + V-cols + engineered + graph."""
    base = ['TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
            'addr1', 'addr2', 'P_emaildomain', 'R_emaildomain']
    C_cols = [c for c in df.columns if c.startswith('C') and c[1:].isdigit()]
    D_cols = [c for c in df.columns if c.startswith('D') and c[1:].isdigit()]
    M_cols = [c for c in df.columns if c.startswith('M') and c[1:].isdigit()]
    extra = ['LogAmt', 'Hour', 'DayOfWeek',
             'R_emaildomain_freq', 'P_emaildomain_freq',
             'Amt_x_Hour', 'C1_div_C14', 'Amt_card_dev',
             'Amt_mean_3', 'Amt_std_3',
             'Amt_cents', 'DT_diff', 'UID_count', 'UID_Amt_mean', 'UID_Amt_std', 'D1n']
    
    # Add card-level agg features
    card_agg_cols = [c for c in df.columns if any(c.startswith(p) for p in ['card1_count', 'card1_amt', 'card2_count', 'card2_amt', 'card5_count', 'card5_amt'])]
    extra.extend(card_agg_cols)
    
    # Add all smoothed target encoded rates
    rate_cols = [c for c in df.columns if c.endswith('_fraud_rate')]
    # === PHASE 6: Include Freq and new Aggs ===
    freq_agg_cols = [c for c in df.columns if c.endswith('_freq') or c.startswith('UID_')]
    extra.extend(freq_agg_cols)
    # Add all new hetero graph features
    hetero_cols = [c for c in df.columns if c.startswith('grp_') or c.startswith('motif_')]
    extra.extend(hetero_cols)
    extra.extend(rate_cols)

    # Add top V-columns (missing rate < 50%)
    # Add PCA components instead of dropping them
    V_good = [c for c in df.columns if c.startswith('V_pca_')]
    if len(V_good) == 0:
        # Fallback to standard V_cols if PCA skipped
        V_cols = [c for c in df.columns if c.startswith('V') and c[1:].isdigit()]
        V_good = [c for c in V_cols if df[c].isna().mean() < 0.50]
        V_good = sorted(V_good, key=lambda c: df[c].isna().mean())[:70]

    all_cols = [c for c in base + C_cols + D_cols + M_cols + extra + V_good if c in df.columns]
    print(f'    Selected {len(all_cols)} features ({len(V_good)} V-columns included)')
    return all_cols

def extract_tabular_view(df, cols):
    print(f'  [View 1] Tabular ({len(cols)} features)')
    return df[cols].values.astype(np.float32)



In [ ]:
SEQ_WINDOW = 10

def extract_sequential_view(df, window=SEQ_WINDOW):
    """T=10 sliding window per AccountID. Right-padded zeros for cuDNN."""
    print(f'  [View 2] Sequential (T={window} per AccountID)...')
    candidates = ['TransactionAmt', 'LogAmt', 'Hour', 'C1', 'C2', 'D1', 'D2',
                  'C14', 'DayOfWeek',
                  'grp_acct_cnt_prev_7d', 'grp_device_cnt_prev_7d', 'grp_pemail_cnt_prev_7d']
    seq_feats = [c for c in candidates if c in df.columns]
    n_feat = len(seq_feats)
    print(f'    Seq features ({n_feat}): {seq_feats}')
    seq = np.zeros((len(df), window, n_feat), dtype=np.float32)
    for _, grp in df.groupby('AccountID'):
        vals = grp[seq_feats].values.astype(np.float32)
        idxs = grp.index.values
        for pos, idx in enumerate(idxs):
            start = max(0, pos - window + 1)
            s = vals[start:pos+1]
            seq[idx, :len(s), :] = s
    print(f'    Shape: {seq.shape}')
    return seq


In [ ]:
def extract_graph_view(df, n_hops=2):
    """Optimized heterogeneous temporal graph-derived features for IEEE-CIS."""
    import time
    from collections import defaultdict, deque

    t0 = time.time()
    print('  [View 3] Optimized Heterogeneous Temporal Graph Features...')

    if 'TransactionDT' not in df.columns:
        raise ValueError('TransactionDT is required for temporal graph features')

    # Preprocessing already sorts by TransactionDT; avoid DataFrame copies here.
    txn_time = df['TransactionDT'].to_numpy(dtype=np.float64, copy=False)
    txn_amt = df['TransactionAmt'].to_numpy(dtype=np.float32, copy=False) if 'TransactionAmt' in df.columns else np.zeros(len(df), dtype=np.float32)
    n = len(df)

    relation_cols = {
        'acct': 'AccountID',
        'card1': 'card1_raw',
        'addr1': 'addr1_raw',
        'pemail': 'P_emaildomain_raw',
        'remail': 'R_emaildomain_raw',
        'device': 'device_fp_raw',
        'product': 'ProductCD_raw',
    }
    unique_counterparts = {
        'card1': ('addr1_raw', 'addr1'),
        'addr1': ('card1_raw', 'card1'),
        'pemail': ('AccountID', 'acct'),
        'remail': ('AccountID', 'acct'),
        'device': ('AccountID', 'acct'),
    }
    windows = {'1d': 86400.0, '7d': 86400.0 * 7, '30d': 86400.0 * 30}
    invalid_tokens = {'UNKNOWN', 'nan', 'None', '-1', ''}

    # Read all relation values once to avoid repeated DataFrame lookups.
    rel_arrays = {
        rel: (df[col].astype(str).to_numpy(copy=False) if col in df.columns else np.full(n, 'UNKNOWN', dtype=object))
        for rel, col in relation_cols.items()
    }

    class WindowState:
        __slots__ = ('d1', 'd7', 'd30', 'sum7', 'cp30')

        def __init__(self, track_cp=False):
            self.d1 = deque()
            self.d7 = deque()
            self.d30 = deque()
            self.sum7 = 0.0
            self.cp30 = {} if track_cp else None

        def expire(self, now, win1, win7, win30):
            while self.d1 and now - self.d1[0][0] > win1:
                self.d1.popleft()
            while self.d7 and now - self.d7[0][0] > win7:
                self.sum7 -= self.d7[0][1]
                self.d7.popleft()
            while self.d30 and now - self.d30[0][0] > win30:
                _, _, cp = self.d30.popleft()
                if self.cp30 is not None:
                    cur = self.cp30.get(cp, 0) - 1
                    if cur <= 0:
                        self.cp30.pop(cp, None)
                    else:
                        self.cp30[cp] = cur

        def add(self, now, amt, cp):
            evt = (now, amt, cp)
            self.d1.append(evt)
            self.d7.append(evt)
            self.d30.append(evt)
            self.sum7 += amt
            if self.cp30 is not None:
                self.cp30[cp] = self.cp30.get(cp, 0) + 1

    states = {
        rel: defaultdict(lambda rel=rel: WindowState(rel in unique_counterparts))
        for rel in relation_cols
    }

    feat = {}
    for rel in relation_cols:
        feat[f'grp_{rel}_cnt_prev_1d'] = np.zeros(n, dtype=np.float32)
        feat[f'grp_{rel}_cnt_prev_7d'] = np.zeros(n, dtype=np.float32)
        feat[f'grp_{rel}_cnt_prev_30d'] = np.zeros(n, dtype=np.float32)
        feat[f'grp_{rel}_amt_sum_7d'] = np.zeros(n, dtype=np.float32)
        feat[f'grp_{rel}_time_since_last'] = np.full(n, windows['30d'], dtype=np.float32)
        if rel in unique_counterparts:
            out_name = unique_counterparts[rel][1]
            feat[f'grp_{rel}_unique_{out_name}_30d'] = np.zeros(n, dtype=np.float32)

    motif_new_device = np.zeros(n, dtype=np.float32)
    motif_new_pemail = np.zeros(n, dtype=np.float32)
    motif_new_addr1 = np.zeros(n, dtype=np.float32)
    motif_pair_card1_addr1_prev_30d = np.zeros(n, dtype=np.float32)
    motif_pair_card1_addr1_unique_acct_30d = np.zeros(n, dtype=np.float32)
    motif_device_unique_addr1_30d = np.zeros(n, dtype=np.float32)
    motif_addr1_unique_device_30d = np.zeros(n, dtype=np.float32)
    motif_acct_burst_ratio_1d_30d = np.zeros(n, dtype=np.float32)

    acct_seen = {
        'device': defaultdict(set),
        'pemail': defaultdict(set),
        'addr1': defaultdict(set),
    }
    acct_hist = defaultdict(int)
    pair_state = defaultdict(lambda: WindowState(track_cp=True))
    device_addr_state = defaultdict(lambda: WindowState(track_cp=True))
    addr_device_state = defaultdict(lambda: WindowState(track_cp=True))

    for idx in range(n):
        now = float(txn_time[idx])
        amt = float(txn_amt[idx])
        acct = rel_arrays['acct'][idx]
        rel_values = {rel: rel_arrays[rel][idx] for rel in relation_cols}

        for rel, key in rel_values.items():
            if key in invalid_tokens:
                continue

            state = states[rel][key]
            state.expire(now, windows['1d'], windows['7d'], windows['30d'])

            feat[f'grp_{rel}_cnt_prev_1d'][idx] = len(state.d1)
            feat[f'grp_{rel}_cnt_prev_7d'][idx] = len(state.d7)
            feat[f'grp_{rel}_cnt_prev_30d'][idx] = len(state.d30)
            feat[f'grp_{rel}_amt_sum_7d'][idx] = state.sum7
            if state.d30:
                feat[f'grp_{rel}_time_since_last'][idx] = now - state.d30[-1][0]
            if rel in unique_counterparts:
                out_name = unique_counterparts[rel][1]
                feat[f'grp_{rel}_unique_{out_name}_30d'][idx] = len(state.cp30)

        card1 = rel_values['card1']
        addr1 = rel_values['addr1']
        device = rel_values['device']

        if card1 not in invalid_tokens and addr1 not in invalid_tokens:
            pair_key = f'{card1}|{addr1}'
            pstate = pair_state[pair_key]
            pstate.expire(now, windows['1d'], windows['7d'], windows['30d'])
            motif_pair_card1_addr1_prev_30d[idx] = len(pstate.d30)
            motif_pair_card1_addr1_unique_acct_30d[idx] = len(pstate.cp30)

        if device not in invalid_tokens:
            dstate = device_addr_state[device]
            dstate.expire(now, windows['1d'], windows['7d'], windows['30d'])
            motif_device_unique_addr1_30d[idx] = len(dstate.cp30)

        if addr1 not in invalid_tokens:
            astate = addr_device_state[addr1]
            astate.expire(now, windows['1d'], windows['7d'], windows['30d'])
            motif_addr1_unique_device_30d[idx] = len(astate.cp30)

        acct_state = states['acct'][acct] if acct not in invalid_tokens else None
        if acct_state is not None:
            denom = float(len(acct_state.d30) + 1)
            motif_acct_burst_ratio_1d_30d[idx] = float(len(acct_state.d1)) / denom

        if acct not in invalid_tokens:
            device = rel_values['device']
            pemail = rel_values['pemail']
            if device not in invalid_tokens and acct_hist[acct] > 0 and device not in acct_seen['device'][acct]:
                motif_new_device[idx] = 1.0
            if pemail not in invalid_tokens and acct_hist[acct] > 0 and pemail not in acct_seen['pemail'][acct]:
                motif_new_pemail[idx] = 1.0
            if addr1 not in invalid_tokens and acct_hist[acct] > 0 and addr1 not in acct_seen['addr1'][acct]:
                motif_new_addr1[idx] = 1.0

        for rel, key in rel_values.items():
            if key in invalid_tokens:
                continue
            cp = 'UNKNOWN'
            if rel in unique_counterparts:
                cp_col = unique_counterparts[rel][0]
                if cp_col in df.columns:
                    cp = str(df.at[idx, cp_col])
            states[rel][key].add(now, amt, cp)

        if card1 not in invalid_tokens and addr1 not in invalid_tokens:
            pair_state[f'{card1}|{addr1}'].add(now, amt, acct if acct not in invalid_tokens else 'UNKNOWN')
        if device not in invalid_tokens:
            device_addr_state[device].add(now, amt, addr1 if addr1 not in invalid_tokens else 'UNKNOWN')
        if addr1 not in invalid_tokens:
            addr_device_state[addr1].add(now, amt, device if device not in invalid_tokens else 'UNKNOWN')

        if acct not in invalid_tokens:
            acct_hist[acct] += 1
            if rel_values['device'] not in invalid_tokens:
                acct_seen['device'][acct].add(rel_values['device'])
            if rel_values['pemail'] not in invalid_tokens:
                acct_seen['pemail'][acct].add(rel_values['pemail'])
            if rel_values['addr1'] not in invalid_tokens:
                acct_seen['addr1'][acct].add(rel_values['addr1'])

    feat['motif_acct_new_device_flag'] = motif_new_device
    feat['motif_acct_new_pemail_flag'] = motif_new_pemail
    feat['motif_acct_new_addr1_flag'] = motif_new_addr1
    feat['motif_pair_card1_addr1_prev_30d'] = motif_pair_card1_addr1_prev_30d
    feat['motif_pair_card1_addr1_unique_acct_30d'] = motif_pair_card1_addr1_unique_acct_30d
    feat['motif_device_unique_addr1_30d'] = motif_device_unique_addr1_30d
    feat['motif_addr1_unique_device_30d'] = motif_addr1_unique_device_30d
    feat['motif_acct_burst_ratio_1d_30d'] = motif_acct_burst_ratio_1d_30d

    gcols = list(feat.keys())
    for col in gcols:
        df[col] = feat[col]

    print(f'    Graph features: {len(gcols)} cols ({time.time()-t0:.1f}s)')
    return df[gcols].values.astype(np.float32), gcols


---
## STAGE 4: Base Models + OOF Predictions (Tier 1+2 Optimized)

| # | Model | View | Imbalance | Optimization |
|---|-------|------|-----------|---------------|
| 1 | XGBoost (800 trees) | Tabular | Native Weights/Focal Loss | L1+L2 reg |
| 2 | LightGBM (800 trees) | Tabular | Native Weights/Focal Loss | L1+L2 reg |


In [ ]:
def focal_loss_tf(gamma=2.0, alpha=0.75):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        at = tf.where(tf.equal(y_true, 1), alpha, 1 - alpha)
        return -tf.reduce_mean(at * tf.pow(1 - pt, gamma) * tf.math.log(pt))
    return loss_fn

def focal_loss_torch(gamma=2.0, alpha=0.75):
    def loss_fn(y_pred, y_true):
        y_pred = torch.clamp(y_pred, 1e-7, 1 - 1e-7)
        pt = torch.where(y_true == 1, y_pred, 1 - y_pred)
        at = torch.where(y_true == 1, alpha, 1 - alpha)
        return -torch.mean(at * (1 - pt) ** gamma * torch.log(pt))


In [ ]:
# MODEL 4: CatBoost (Tabular View — excels at categorical features) — Tier 2
def train_catboost(X_tr, y_tr):
    cat_iters = globals().get('CAT_ITERS', 800)
    cat_depth = globals().get('CAT_DEPTH', 8)
    cat_lr = globals().get('CAT_LR', 0.03)
    use_gpu = globals().get('USE_GPU_FOR_CATBOOST', False)
    print(f"    [CatBoost] Training ({cat_iters} iterations, {'GPU' if use_gpu else 'CPU'})...")
    cb = CatBoostClassifier(
        iterations=cat_iters, depth=cat_depth, learning_rate=cat_lr,
        task_type='GPU' if use_gpu else 'CPU',
        devices='0' if use_gpu else None,
        auto_class_weights='Balanced',
        l2_leaf_reg=3.0,
        verbose=0, random_seed=42
    )
    cb.fit(X_tr, y_tr)
    return cb


In [ ]:

class TabMWrapper:
    def __init__(self, model, device, batch_size=4096):
        self.model = model
        self.device = device
        self.batch_size = batch_size

    def predict_proba(self, X_tab):
        X_np = np.asarray(X_tab, dtype=np.float32)
        preds = []
        self.model.eval()
        with torch.no_grad():
            for i in range(0, len(X_np), self.batch_size):
                xb = torch.from_numpy(X_np[i:i+self.batch_size]).to(self.device, non_blocking=(self.device.type == 'cuda'))
                logits = self.model(xb)
                probs = torch.sigmoid(logits).mean(dim=1).cpu().numpy()
                preds.append(probs)
        p1 = np.concatenate(preds) if preds else np.zeros((0,), dtype=np.float32)
        return np.column_stack([1.0 - p1, p1]).astype(np.float32)


def get_base_model_names():
    names = []
    run_lgb_only = globals().get('RUN_LGB_ONLY', False)
    if globals().get('USE_XGB_MODEL', True) and not run_lgb_only:
        names.append('XGB')
    names.append('LGB')
    if globals().get('USE_CATBOOST_MODEL', True) and not run_lgb_only:
        names.append('CatBoost')
    if globals().get('USE_TABM_MODEL', False) and not run_lgb_only:
        names.append('TabM')
    return names


def train_tabm_model(X_tab_tr, y_tr):
    from tabm.tabm_model import TabMStyleModel

    use_gpu = globals().get('USE_GPU_FOR_TABM', True)
    epochs = globals().get('TABM_EPOCHS', 6)
    batch_size = globals().get('TABM_BATCH_SIZE', 2048)
    d_token = globals().get('TABM_D_TOKEN', 16)
    hidden_dim = globals().get('TABM_HIDDEN_DIM', 256)
    n_layers = globals().get('TABM_N_LAYERS', 3)
    n_members = globals().get('TABM_N_MEMBERS', 4)
    lr = globals().get('TABM_LR', 1e-3)

    device = torch.device('cuda' if use_gpu and torch.cuda.is_available() else 'cpu')
    X_np = np.asarray(X_tab_tr, dtype=np.float32)
    y_np = np.asarray(y_tr, dtype=np.float32)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X_np), torch.from_numpy(y_np))
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True, pin_memory=(device.type == 'cuda'))

    model = TabMStyleModel(
        num_features=X_np.shape[1], d_token=d_token, hidden_dim=hidden_dim,
        n_layers=n_layers, n_members=n_members
    ).to(device)

    pos = max(float(y_np.sum()), 1.0)
    neg = max(float(len(y_np) - y_np.sum()), 1.0)
    pos_weight = torch.tensor([neg / pos], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(epochs):
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=(device.type == 'cuda'))
            yb = yb.to(device, non_blocking=(device.type == 'cuda'))
            opt.zero_grad()
            logits = model(xb)
            target = yb.unsqueeze(1).expand_as(logits)
            loss = criterion(logits, target)
            loss.backward()
            opt.step()

    return TabMWrapper(model=model, device=device, batch_size=batch_size)


def generate_oof_train(X_tab_tr, y_tr):
    """Train base models for stacking. Returns trained models."""
    ratio = float((y_tr==0).sum()) / max(float((y_tr==1).sum()), 1.0)
    run_lgb_only = globals().get('RUN_LGB_ONLY', False)
    use_xgb = globals().get('USE_XGB_MODEL', True) and not run_lgb_only
    use_cat = globals().get('USE_CATBOOST_MODEL', True) and not run_lgb_only
    use_tabm = globals().get('USE_TABM_MODEL', False) and not run_lgb_only
    xgb_trees = globals().get('XGB_TREES', 800)
    xgb_depth = globals().get('XGB_MAX_DEPTH', 8)
    use_xgb_gpu = globals().get('USE_GPU_FOR_XGB', True)
    use_lgb_gpu = globals().get('USE_GPU_FOR_LGB', False)
    lgb_trees = globals().get('LGB_TREES', 800)
    lgb_depth = globals().get('LGB_MAX_DEPTH', 10)
    lgb_num_leaves = globals().get('LGB_NUM_LEAVES', 63)
    lgb_min_child_samples = globals().get('LGB_MIN_CHILD_SAMPLES', 20)
    lgb_scale_pos_weight_mult = globals().get('LGB_SCALE_POS_WEIGHT_MULT', 1.0)
    xc = None
    if use_xgb and xgb_trees > 0:
        print(f'    [XGB] Training ({xgb_trees} trees)...')
        xc = xgb.XGBClassifier(n_estimators=xgb_trees, max_depth=xgb_depth, learning_rate=0.03,
            scale_pos_weight=ratio,
            tree_method='hist', device='cuda' if use_xgb_gpu else 'cpu',
            colsample_bytree=0.7, subsample=0.8,
            reg_alpha=0.1, reg_lambda=1.0, min_child_weight=5, gamma=0.1,
            random_state=42, n_jobs=-1)
        xc.fit(X_tab_tr, y_tr)
    else:
        print('    [XGB] Skipped by config.')

    print(f"    [LGB] Training ({lgb_trees} trees, {'GPU' if use_lgb_gpu else 'CPU'}, leaves={lgb_num_leaves}, min_child={lgb_min_child_samples}, spw_mult={lgb_scale_pos_weight_mult})...")
    lc = lgb.LGBMClassifier(n_estimators=lgb_trees, max_depth=lgb_depth, learning_rate=0.03,
        scale_pos_weight=ratio * lgb_scale_pos_weight_mult,
        colsample_bytree=0.7, subsample=0.8,
        reg_alpha=0.1, reg_lambda=1.0, min_child_samples=lgb_min_child_samples, num_leaves=lgb_num_leaves,
        device='gpu' if use_lgb_gpu else 'cpu',
        random_state=42, n_jobs=-1, verbose=-1)
    lc.fit(X_tab_tr, y_tr)

    cb = train_catboost(X_tab_tr, y_tr) if use_cat else None
    if not use_cat:
        print('    [CatBoost] Skipped by config.')

    tm = None
    if use_tabm:
        print(f"    [TabM] Training ({globals().get('TABM_EPOCHS', 6)} epochs, {'GPU' if globals().get('USE_GPU_FOR_TABM', True) else 'CPU'})...")
        tm = train_tabm_model(X_tab_tr, y_tr)
    else:
        print('    [TabM] Skipped by config.')

    return xc, lc, cb, tm


def predict_oof(models, X_tab):
    """Predict with configured base models. No re-training."""
    xc, lc, cb, tm = models
    active_names = get_base_model_names()
    model_map = {'XGB': xc, 'LGB': lc, 'CatBoost': cb, 'TabM': tm}
    n = X_tab.shape[0]
    oof = np.zeros((n, len(active_names)), dtype=np.float32)
    for j, name in enumerate(active_names):
        model = model_map[name]
        if model is not None:
            oof[:, j] = model.predict_proba(X_tab)[:,1]
    print(f'    OOF shape: {oof.shape}')
    return oof, lc


---
## STAGE 5: Meta-Learner
$$\hat{Y}_i = \sigma\left(\sum_{m=1}^{3} \omega_m \cdot \hat{y}_i^{(m)} + \beta\right)$$


In [ ]:

def stage5_meta(oof_tr, y_tr, oof_vl):
    names = get_base_model_names()
    meta_name = globals().get('META_LEARNER', 'xgb')
    enable_cal = globals().get('ENABLE_CALIBRATION', True)
    cal_method = globals().get('CALIBRATION_METHOD', 'isotonic')
    cal_holdout = globals().get('CALIBRATION_HOLDOUT', 0.20)

    def build_meta_model():
        if meta_name == 'logreg':
            c_val = globals().get('LOGREG_C', 1.0)
            return LogisticRegression(
                penalty='l2', C=c_val, solver='liblinear',
                class_weight='balanced', max_iter=1000, random_state=42
            )
        use_xgb_gpu = globals().get('USE_GPU_FOR_XGB', True)
        return xgb.XGBClassifier(
            n_estimators=50, max_depth=3, learning_rate=0.1,
            scale_pos_weight=1.0, eval_metric='aucpr',
            tree_method='hist', device='cuda' if use_xgb_gpu else 'cpu',
            random_state=42, n_jobs=-1
        )

    label = 'LogReg L2' if meta_name == 'logreg' else 'XGB'
    print(f'[Stage 5] Meta-Learner ({label}) on OOF [Nx{len(names)}]...')

    m = build_meta_model()
    m.fit(oof_tr, y_tr)
    raw_preds = m.predict_proba(oof_vl)[:,1]

    if meta_name == 'logreg':
        weights = m.coef_[0]
        print(f'  Coefficients: {dict(zip(names, np.round(weights, 4)))}')
    else:
        importances = m.feature_importances_
        print(f'  Feature Importances: {dict(zip(names, importances.round(4)))}')

    if not enable_cal or len(np.unique(y_tr)) < 2 or len(y_tr) < 1000:
        return m, raw_preds

    tr_meta, cal_meta, y_meta, y_cal = train_test_split(
        oof_tr, y_tr, test_size=cal_holdout, stratify=y_tr, random_state=42
    )
    m_cal = build_meta_model()
    m_cal.fit(tr_meta, y_meta)
    cal_raw = m_cal.predict_proba(cal_meta)[:,1]
    vl_raw = m_cal.predict_proba(oof_vl)[:,1]

    if cal_method == 'platt':
        calibrator = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
        calibrator.fit(cal_raw.reshape(-1, 1), y_cal)
        preds = calibrator.predict_proba(vl_raw.reshape(-1, 1))[:,1]
        cal_fit = calibrator.predict_proba(cal_raw.reshape(-1, 1))[:,1]
    else:
        calibrator = IsotonicRegression(out_of_bounds='clip')
        calibrator.fit(cal_raw, y_cal)
        preds = calibrator.transform(vl_raw)
        cal_fit = calibrator.transform(cal_raw)

    raw_brier = brier_score_loss(y_cal, cal_raw)
    cal_brier = brier_score_loss(y_cal, cal_fit)
    print(f'  [Calibration] {cal_method} holdout={cal_holdout:.2f}, Brier {raw_brier:.4f} -> {cal_brier:.4f}')
    return m, preds


---
## STAGE 6: 4-Level XAI Framework
| Tool | Purpose | Metric |
|------|---------|--------|
| SHAP | Feature importance | FRD |
| LIME | Local surrogate | R² fidelity |
| DiCE | Counterfactual recourse | Validity % |


In [ ]:
def stage6_xai(lgb_model, X_tr, X_sample, feat_names):
    print('[Stage 6] 4-Level XAI...')
    top3 = []

    # 1. SHAP
    print('  [SHAP] TreeExplainer...')
    exp = shap.TreeExplainer(lgb_model)
    sv = exp.shap_values(X_sample[:100])
    sv_f = sv[1] if isinstance(sv, list) else sv
    shap.summary_plot(sv_f, X_sample[:100], feature_names=feat_names, plot_type='dot', show=True)
    t3 = np.argsort(np.abs(sv_f[0]))[-3:][::-1]
    top3 = [(feat_names[i], round(sv_f[0][i], 4)) for i in t3]
    print(f'    Top-3: {top3}')

    # 2. LIME
    print('  [LIME] Local Surrogate...')
    le = lime.lime_tabular.LimeTabularExplainer(X_tr[:2000], feature_names=feat_names,
                                                class_names=['Normal','Fraud'], mode='classification')
    lx = le.explain_instance(X_sample[0], lgb_model.predict_proba, num_features=10, num_samples=10000)
    print(f'    LIME R^2: {lx.score:.4f}')
    lx.show_in_notebook()

    # 3. DiCE
    print('  [DiCE] Counterfactuals...')
    try:
        df_tr = pd.DataFrame(X_tr[:300], columns=feat_names)
        df_s = pd.DataFrame(X_sample[:3], columns=feat_names)
        d = dice_ml.Data(dataframe=df_tr.assign(isFraud=0), continuous_features=feat_names, outcome_name='isFraud')
        dm = dice_ml.Model(model=lgb_model, backend='sklearn')
        de = dice_ml.Dice(d, dm, method='random')
        cf = de.generate_counterfactuals(df_s.iloc[[0]], total_CFs=3, desired_class='opposite')
        cf.visualize_as_dataframe(show_only_changes=True)
        print('    DiCE generated.')
    except Exception as e:
        print(f'    DiCE skipped: {e}')

    # 4. Anchors
    print('  [Anchors] IF-THEN Rules...')
    try:
        from alibi.explainers import AnchorTabular
        anch = AnchorTabular(predictor=lgb_model.predict, feature_names=feat_names)
        anch.fit(X_tr[:2000], disc_perc=(10, 25, 50, 75, 90))
        explanation = anch.explain(X_sample[0])
        print(f'    Rule: {" AND ".join(explanation.anchor)}')
        print(f'    Precision: {explanation.precision:.4f}, Coverage: {explanation.coverage:.4f}')
    except Exception as e:
        print(f'    Anchors skipped: {e}')

    return top3


---


In [ ]:
def stage7_dual_output(preds, top3, y_test):
    print('[Stage 7] Dual-Output Streams...')
    dec = np.where(preds >= 0.5, 'BLOCK', 'ALLOW')
    rt = pd.DataFrame({'FraudScore': preds.round(4), 'CI_Lo': (preds-0.05).round(4),
                       'CI_Hi': (preds+0.05).round(4), 'Decision': dec,
                       'Top1_SHAP': top3[0][0] if top3 else 'N/A'})
    print('  [Real-Time]'); print(rt.head(10).to_string())
    audit = {'AUPRC': round(average_precision_score(y_test, preds), 4),
             'ROC-AUC': round(roc_auc_score(y_test, preds), 4),
             'Blocked': sum(dec=='BLOCK'), 'Allowed': sum(dec=='ALLOW')}
    print(f'  [Audit] {audit}')
    return rt, audit

def stage8_hitl(preds, df_test):
    print('[Stage 8] HITL Escalation + Regulatory...')
    unc = (preds >= 0.5) & (preds < 0.7)
    amt_col = 'TransactionAmt' if 'TransactionAmt' in df_test.columns else None
    if amt_col:
        hv = df_test[amt_col].values[:len(preds)] > df_test[amt_col].quantile(0.95)
    else:
        hv = np.zeros(len(preds), dtype=bool)
    esc = unc | hv
    print(f'  Uncertain: {unc.sum()}, High-value: {hv.sum()}, HITL Total: {esc.sum()}')
    for reg, tool in [('GDPR Art.22','LIME+DiCE'), ('EU AI Act','Full audit'),
                      ('Basel III','SHAP+McNemar'), ('EU AI Act Art.14',f'{esc.sum()} txns to HITL')]:
        print(f'    [{reg}] {tool}')


---


In [ ]:
# ====== DATA LOADING & EXTRACTION (ZIP MODE — FULL DATA) ======
import os
import pandas as pd
import numpy as np
import zipfile
import glob
import gc

ZIP_PATH = '/content/drive/MyDrive/MVS_XAI_Data/ieee-fraud-detection.zip'
EXTRACT_DIR = '/content/ieee-dataset'

print('--- Full Data Loading (IEEE-CIS 590K) ---')
print(f'Checking for ZIP file: {ZIP_PATH}')

df_txn = None
df_id = None

if os.path.exists(ZIP_PATH):
    print('\u2705 Found ZIP file! Extracting to local Colab storage...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    print('\u2705 Extraction complete!')

    txn_files = glob.glob(f'{EXTRACT_DIR}/**/train_transaction.csv', recursive=True)
    id_files  = glob.glob(f'{EXTRACT_DIR}/**/train_identity.csv', recursive=True)

    if txn_files:
        print(f'\u2705 Loading FULL train_transaction (no row limit)...')
        df_txn = pd.read_csv(txn_files[0])
        # Convert to float32 immediately to save ~50% RAM
        for col in df_txn.select_dtypes(include=['float64']).columns:
            df_txn[col] = df_txn[col].astype(np.float32)
        print(f'    Shape: {df_txn.shape}, Memory: {df_txn.memory_usage(deep=True).sum()/1e9:.2f} GB')
    else:
        print('\u274c train_transaction.csv not found in ZIP!')

    if id_files:
        print(f'\u2705 Loading train_identity...')
        df_id = pd.read_csv(id_files[0])
        for col in df_id.select_dtypes(include=['float64']).columns:
            df_id[col] = df_id[col].astype(np.float32)
    else:
        print('\u274c train_identity.csv not found in ZIP!')
else:
    print('\u274c ZIP file MISSING at MVS_XAI_Data/ieee-fraud-detection.zip')

print('\n============================================================')
if df_txn is not None:
    if df_id is not None:
        df_raw = df_txn.merge(df_id, on='TransactionID', how='left')
        del df_txn, df_id
        gc.collect()
        print(f'Merged data shape: {df_raw.shape}')
    else:
        df_raw = df_txn
        del df_txn
        gc.collect()
        print(f'Transaction-only shape: {df_raw.shape}')
    print(f'Total Memory: {df_raw.memory_usage(deep=True).sum()/1e9:.2f} GB')
else:
    print('IEEE-CIS dataset NOT FOUND! Falling back to synthetic data...')
    print('============================================================\n')
    np.random.seed(42)
    N = 20000
    df_raw = pd.DataFrame({
        'TransactionID': range(N),
        'TransactionDT': np.sort(np.random.randint(100000, 16000000, N)),
        'TransactionAmt': np.random.uniform(1, 5000, N).astype(np.float32),
        'ProductCD': np.random.choice(['W','H','C','S','R'], N),
        'card1': np.random.randint(1000, 9999, N),
        'card2': np.random.randint(100, 600, N).astype(np.float32),
        'card3': np.random.randint(100, 250, N).astype(np.float32),
        'card4': np.random.choice(['visa','mastercard','discover'], N),
        'card5': np.random.randint(100, 300, N).astype(np.float32),
        'card6': np.random.choice(['debit','credit'], N),
        'addr1': np.random.randint(100, 500, N).astype(np.float32),
        'addr2': np.random.randint(1, 100, N).astype(np.float32),
        'P_emaildomain': np.random.choice(['gmail.com','yahoo.com','outlook.com', np.nan], N),
        'R_emaildomain': np.random.choice(['gmail.com','yahoo.com', np.nan], N),
        'isFraud': np.random.choice([0, 1], N, p=[0.965, 0.035])
    })
    print(f'Synthetic Dataset shape: {df_raw.shape}\n')

print(df_raw.head())



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score
import time as _time

FAST_COLAB_T4 = True
ALL_GPU_MODE = True  # Screening on the working GPU path.
USE_STRICT_TIME_SPLIT = True  # Full confirmation should use temporal evaluation.
ENABLE_GRAPH_VIEW = True
ENABLE_SEQ_VIEW = False
ENABLE_UID_SMOOTHING = True
RUN_LGB_ONLY = False
USE_XGB_MODEL = True
USE_CATBOOST_MODEL = True
USE_TABM_MODEL = True
USE_GPU_FOR_XGB = True
USE_GPU_FOR_LGB = ALL_GPU_MODE
USE_GPU_FOR_CATBOOST = ALL_GPU_MODE
USE_GPU_FOR_TABM = ALL_GPU_MODE
META_LEARNER = 'xgb'  # options: logreg, xgb
LOGREG_C = 1.0
ENABLE_CALIBRATION = False
CALIBRATION_METHOD = 'isotonic'  # options: isotonic, platt
CALIBRATION_HOLDOUT = 0.20
THRESHOLD_POLICY = 'best_f1'  # options: best_f1, best_f2, recall_at_precision
THRESHOLD_BETA = 2.0
THRESHOLD_MIN_PRECISION = 0.70

OUTER_FOLDS = 3 if FAST_COLAB_T4 else 5
INNER_FOLDS = 2 if FAST_COLAB_T4 else 3
XGB_TREES = 300
XGB_MAX_DEPTH = 8
LGB_TREES = 300
LGB_MAX_DEPTH = 8
LGB_NUM_LEAVES = 63
LGB_MIN_CHILD_SAMPLES = 20
LGB_SCALE_POS_WEIGHT_MULT = 1.00
CAT_ITERS = 300
CAT_DEPTH = 8
CAT_LR = 0.03
TABM_EPOCHS = 6 if FAST_COLAB_T4 else 10
TABM_BATCH_SIZE = 2048
TABM_D_TOKEN = 16
TABM_HIDDEN_DIM = 256
TABM_N_LAYERS = 3
TABM_N_MEMBERS = 4
TABM_LR = 1e-3

print(f'[Config] FAST_COLAB_T4={FAST_COLAB_T4}, ALL_GPU_MODE={ALL_GPU_MODE}, OUTER_FOLDS={OUTER_FOLDS}, INNER_FOLDS={INNER_FOLDS}, ENABLE_GRAPH_VIEW={ENABLE_GRAPH_VIEW}, ENABLE_SEQ_VIEW={ENABLE_SEQ_VIEW}, RUN_LGB_ONLY={RUN_LGB_ONLY}, USE_XGB_MODEL={USE_XGB_MODEL}, USE_CATBOOST_MODEL={USE_CATBOOST_MODEL}, USE_TABM_MODEL={USE_TABM_MODEL}, ENABLE_UID_SMOOTHING={ENABLE_UID_SMOOTHING}, META_LEARNER={META_LEARNER}, LOGREG_C={LOGREG_C}, ENABLE_CALIBRATION={ENABLE_CALIBRATION}, CALIBRATION_METHOD={CALIBRATION_METHOD}, CALIBRATION_HOLDOUT={CALIBRATION_HOLDOUT}, THRESHOLD_POLICY={THRESHOLD_POLICY}, THRESHOLD_MIN_PRECISION={THRESHOLD_MIN_PRECISION}, LGB_TREES={LGB_TREES}, LGB_MAX_DEPTH={LGB_MAX_DEPTH}, LGB_NUM_LEAVES={LGB_NUM_LEAVES}, LGB_MIN_CHILD_SAMPLES={LGB_MIN_CHILD_SAMPLES}, LGB_SCALE_POS_WEIGHT_MULT={LGB_SCALE_POS_WEIGHT_MULT}, USE_GPU_FOR_XGB={USE_GPU_FOR_XGB}, USE_GPU_FOR_LGB={USE_GPU_FOR_LGB}, USE_GPU_FOR_CATBOOST={USE_GPU_FOR_CATBOOST}, USE_GPU_FOR_TABM={USE_GPU_FOR_TABM}, TABM_EPOCHS={TABM_EPOCHS}')
# ====== STAGE 1 ======
t_start = _time.time()
df_proc = stage1_preprocessing(df_raw)
del df_raw; import gc; gc.collect()

# ====== STAGE 3 (Fold-local feature transforms happen inside CV) ======
print('\n[Stage 3] Multi-View Feature Engineering...')
if ENABLE_GRAPH_VIEW:
    print('  [View 3] Graph view enabled. Fold-local graph features will be built inside each CV fold.')
else:
    print('  [View 3] Graph view skipped by config.')
if ENABLE_SEQ_VIEW:
    print('  [View 2 WARNING] Sequence view is disabled in fold-local mode for now.')
X_seq = None
y = df_proc['isFraud'].values
uid_array = df_proc['AccountID'].values  # PHASE 6: UID Extraction
df_meta = df_proc[['TransactionAmt']].copy() if 'TransactionAmt' in df_proc.columns else None
print('  Base dataframe ready. Fold-local transforms will be fit inside each CV fold.')

# ====== STAGE 2: Cross-Validation ======
print('\n[Stage 2] Cross-Validation...')
if USE_STRICT_TIME_SPLIT:
    print(f'  [Validation] Using {OUTER_FOLDS}-Block Walk-Forward CV (Strict/Realistic)\n')
    cv = TimeSeriesSplit(n_splits=OUTER_FOLDS)
else:
    print(f'  [Validation] Using {OUTER_FOLDS}-Fold Stratified CV (Randomized/High Metrics)\n')
    from sklearn.model_selection import StratifiedKFold
    cv = StratifiedKFold(n_splits=OUTER_FOLDS, shuffle=True, random_state=42)
fold_metrics = []

for fold, (tr_i, vl_i) in enumerate(cv.split(np.zeros(len(y)), y)):
    t_fold = _time.time()
    print(f'\n{"="*60}\nFOLD {fold+1}/{OUTER_FOLDS}\n{"="*60}')
    df_tr_raw = df_proc.iloc[tr_i].reset_index(drop=True).copy()
    df_vl_raw = df_proc.iloc[vl_i].reset_index(drop=True).copy()
    df_tr_fold, df_vl_fold = fit_apply_fold_preprocessing(df_tr_raw, df_vl_raw)
    selected = select_tabular_cols(df_tr_fold)
    Xt_tr = extract_tabular_view(df_tr_fold, selected)
    Xt_vl = extract_tabular_view(df_vl_fold, selected)
    y_tr, y_vl = y[tr_i], y[vl_i]

    from sklearn.model_selection import StratifiedKFold as SKF_inner
    inner_cv = SKF_inner(n_splits=INNER_FOLDS, shuffle=True, random_state=42)
    n_tr = len(y_tr)
    oof_train = np.zeros((n_tr, 3), dtype=np.float32)
    print(f'  [Full-OOF] Generating OOF on {n_tr} samples via {INNER_FOLDS}-fold internal CV...')
    
    for ifold, (itr, ivl) in enumerate(inner_cv.split(np.zeros(n_tr), y_tr)):
        print(f'    [Inner {ifold+1}/{INNER_FOLDS}] Train: {len(itr)}, Val: {len(ivl)}')
        inner_tr_raw = df_tr_raw.iloc[itr].reset_index(drop=True).copy()
        inner_vl_raw = df_tr_raw.iloc[ivl].reset_index(drop=True).copy()
        inner_tr_fold, inner_vl_fold = fit_apply_fold_preprocessing(inner_tr_raw, inner_vl_raw)
        inner_selected = select_tabular_cols(inner_tr_fold)
        Xt_inner_tr = extract_tabular_view(inner_tr_fold, inner_selected)
        Xt_inner_vl = extract_tabular_view(inner_vl_fold, inner_selected)
        inner_models = generate_oof_train(Xt_inner_tr, y_tr[itr])
        oof_part, _ = predict_oof(inner_models, Xt_inner_vl)
        oof_train[ivl] = oof_part
        del inner_models, inner_tr_fold, inner_vl_fold, Xt_inner_tr, Xt_inner_vl; import gc; gc.collect()
    
    print('  [Stage 4] Training FINAL models on FULL training fold...')
    models = generate_oof_train(Xt_tr, y_tr)
    
    print('  [Stage 4] Predicting OOF on validation...')
    oof_vl, best_lgb = predict_oof(models, Xt_vl)

    # Stage 5: scoring / meta-learner
    if RUN_LGB_ONLY:
        meta = None
        meta_p = oof_vl[:,1].copy()
        print('[Stage 5] Meta-Learner skipped: using raw LGB probabilities (RUN_LGB_ONLY=True).')
    else:
        meta, meta_p = stage5_meta(oof_train, y_tr, oof_vl)
    import pandas as pd
    if ENABLE_UID_SMOOTHING:
        uid_vl = uid_array[vl_i]
        uid_df = pd.DataFrame({'UID': uid_vl, 'pred': meta_p})
        uid_mean = uid_df.groupby('UID')['pred'].transform('mean')
        meta_p = 0.7 * meta_p + 0.3 * uid_mean.values
        print('  [Stage 5b] UID smoothing applied.')
    else:
        print('  [Stage 5b] UID smoothing skipped by config.')
    auprc = average_precision_score(y_vl, meta_p)
    roc = roc_auc_score(y_vl, meta_p)

    threshold_grid = np.arange(0.1, 0.9, 0.05)
    threshold_rows = []
    for t in threshold_grid:
        yt = (meta_p >= t).astype(int)
        p_t = precision_score(y_vl, yt, zero_division=0)
        r_t = recall_score(y_vl, yt, zero_division=0)
        f1_t = f1_score(y_vl, yt, zero_division=0)
        beta = globals().get('THRESHOLD_BETA', 2.0)
        fbeta_t = 0.0 if (p_t == 0 and r_t == 0) else ((1 + beta**2) * p_t * r_t) / max(beta**2 * p_t + r_t, 1e-12)
        threshold_rows.append((t, p_t, r_t, f1_t, fbeta_t))

    best_t, best_f1, best_p, best_r = 0.5, 0, 0, 0
    policy = globals().get('THRESHOLD_POLICY', 'best_f1')
    min_prec = globals().get('THRESHOLD_MIN_PRECISION', 0.70)
    if policy == 'recall_at_precision':
        candidates = [row for row in threshold_rows if row[1] >= min_prec]
        chosen = max(candidates, key=lambda row: row[2], default=max(threshold_rows, key=lambda row: row[2]))
        best_t, best_p, best_r, best_f1 = chosen[0], chosen[1], chosen[2], chosen[3]
    elif policy == 'best_f2':
        chosen = max(threshold_rows, key=lambda row: row[4])
        best_t, best_p, best_r, best_f1 = chosen[0], chosen[1], chosen[2], chosen[3]
    else:
        chosen = max(threshold_rows, key=lambda row: row[3])
        best_t, best_p, best_r, best_f1 = chosen[0], chosen[1], chosen[2], chosen[3]

    y_pred = (meta_p >= best_t).astype(int)
    f1 = best_f1; prec = best_p; rec = best_r
    print(f'  [Threshold Policy] {policy} -> t={best_t:.2f}, F1={f1:.4f}, P={prec:.4f}, R={rec:.4f}')

    fold_time = _time.time() - t_fold
    fold_metrics.append({'Fold': fold+1, 'AUPRC': auprc, 'ROC-AUC': roc,
                         'F1': f1, 'Precision': prec, 'Recall': rec,
                         'Time_min': fold_time/60})
    print(f'  FOLD {fold+1}: AUPRC={auprc:.4f}, ROC-AUC={roc:.4f}, F1={f1:.4f}, P={prec:.4f}, R={rec:.4f} ({fold_time/60:.1f}min)')
    import gc; gc.collect()

    if fold == OUTER_FOLDS - 1:
        print(f'\n{"="*60}\nFOLD {OUTER_FOLDS} (FINAL): Detailed Metrics + XAI\n{"="*60}')
        print('\n--- Classification Report ---')
        print(classification_report(y_vl, y_pred, target_names=['Normal', 'Fraud']))
        print('--- Confusion Matrix ---')
        cm = confusion_matrix(y_vl, y_pred)
        print(f'  TN={cm[0,0]:,}  FP={cm[0,1]:,}')
        print(f'  FN={cm[1,0]:,}  TP={cm[1,1]:,}')
        print(f'\n--- Threshold Analysis ---')
        for t, p_t, r_t, f1_t, fbeta_t in threshold_rows:
            print(f'  t={t:.2f}: F1={f1_t:.4f}, P={p_t:.4f}, R={r_t:.4f}, F{THRESHOLD_BETA:.0f}={fbeta_t:.4f}')

        # XAI
        Xs = Xt_vl[:200]
        top3 = stage6_xai(best_lgb, Xt_tr[:2000], Xs, selected)
        stage7_dual_output(meta_p, top3, y_vl)
        if df_meta is not None:
            stage8_hitl(meta_p, df_meta.iloc[vl_i])

        # === VISUALIZATION ===
        try:
            import matplotlib.pyplot as plt
            import seaborn as sns
            from sklearn.metrics import roc_curve, precision_recall_curve
            print('\n--- Visualizing Results ---')
            fig, axes = plt.subplots(1, 4, figsize=(24, 5))
            sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0])
            axes[0].set_title('Confusion Matrix (Final Fold)')
            axes[0].set_xlabel('Predicted')
            axes[0].set_ylabel('Actual')
            fpr, tpr, _ = roc_curve(y_vl, meta_p)
            axes[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc:.4f})')
            axes[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
            axes[1].set_title('ROC Curve')
            axes[1].set_xlabel('False Positive Rate')
            axes[1].set_ylabel('True Positive Rate')
            axes[1].legend(loc='lower right')
            prec_c, rec_c, _ = precision_recall_curve(y_vl, meta_p)
            axes[2].plot(rec_c, prec_c, color='purple', lw=2, label=f'PRC (AUPRC = {auprc:.4f})')
            axes[2].set_title('Precision-Recall Curve')
            axes[2].set_xlabel('Recall')
            axes[2].set_ylabel('Precision')
            axes[2].legend(loc='lower left')
            models_names = get_base_model_names()
            if meta is None:
                weights = np.zeros(len(models_names), dtype=float)
            elif hasattr(meta, 'feature_importances_'):
                weights = meta.feature_importances_
            elif hasattr(meta, 'coef_'):
                weights = np.abs(meta.coef_[0])
            else:
                weights = np.zeros(len(models_names), dtype=float)
            sns.barplot(x=models_names, y=weights, palette='viridis', ax=axes[3])
            axes[3].set_title(f'Meta-Learner Weights ({len(models_names)} Models)')
            axes[3].set_ylabel('Importance')
            plt.tight_layout()
            plt.show()
        except Exception as e:
            print(f'Visualization error: {e}')


# ====== SUMMARY ======
total_time = (_time.time() - t_start) / 60
print(f'\n{"="*60}\nMVS-XAI IEEE-CIS Pipeline Complete (Tier 3 Phase1 — {len(get_base_model_names())} Models + XGB Meta)\n{"="*60}')
mdf = pd.DataFrame(fold_metrics)
print(mdf.to_string(index=False))
print(f'\nMean AUPRC:   {mdf["AUPRC"].mean():.4f} +/- {mdf["AUPRC"].std():.4f}')
print(f'Mean ROC-AUC: {mdf["ROC-AUC"].mean():.4f} +/- {mdf["ROC-AUC"].std():.4f}')
print(f'Mean F1:      {mdf["F1"].mean():.4f} +/- {mdf["F1"].std():.4f}')
print(f'Mean Prec:    {mdf["Precision"].mean():.4f} +/- {mdf["Precision"].std():.4f}')
print(f'Mean Recall:  {mdf["Recall"].mean():.4f} +/- {mdf["Recall"].std():.4f}')
print(f'Total runtime: {total_time:.1f} min (FAST_COLAB_T4={FAST_COLAB_T4})')


